# 04. PSA da v1.1: métricas, tabela por termo e bootstrap

Contribuição B (análise estatística). A v1.1 é o corpus do WICS sem as sentenças repetidas (2.730 pares, 130 contextos, 21 termos).

- **Entrada:** `data_bits/BITS_PTBR_v1.1_pareado_com_scores.csv` (gerado no notebook 03).
- **Saídas usadas no artigo:** `results/b_tabela_por_termo_v1.1.csv` (usada no notebook 05) e `results/BOOTSTRAP_V1_1_COMPLETO.csv` (ScoreSense, LabelDist, FlipRate e ScoreDev com IC95%, por contexto e por contexto + termo).

As métricas binárias deste notebook usam o limiar 0,5; a sensibilidade ao limiar está no notebook 03. O notebook reúne etapas executadas em sessões diferentes: as células de ScoreDev da seção 1 usam o dataframe carregado na seção 2.

**Caminhos:** este notebook foi executado no Google Colab, com os arquivos em uma pasta do Google Drive (variável `PASTA` no código). No repositório, as entradas estão em `data_bits/` e as saídas em `results/`. Para reexecutar, ajuste `PASTA`.

## 1. ScoreDev por grupo

### Função do ScoreDev

Para cada contexto (`pair_id`), desvio padrão dos scores das frases com cada termo; depois, a média entre os contextos (Venkit et al., 2023). Usa os scores das frases com o termo, não os deltas, e não inclui o controle.

In [ ]:
import numpy as np
import pandas as pd

def calculate_scoredev(df, score_col, model_name):

    required_cols = ["pair_id", "Class", "SubClass", score_col]

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        raise ValueError(
            f"Colunas ausentes para {model_name}: {missing}"
        )

    if df[score_col].isna().any():
        raise ValueError(
            f"Existem valores NaN em {score_col}"
        )

    scoredev_by_pair = (
        df
        .groupby("pair_id")
        .agg(
            ScoreDev_context=(score_col, lambda x: np.std(x, ddof=0)),
            n_perturbations=(score_col, "count")
        )
        .reset_index()
    )

    scoredev_global = (
        scoredev_by_pair["ScoreDev_context"].mean()
    )

    scoredev_by_pair["Model"] = model_name

    return scoredev_global, scoredev_by_pair

### ScoreDev por grupo: BERTimbau toxic

In [ ]:
import numpy as np
import pandas as pd

scoredev_tox1_context = (
    df_bits_paired
    .groupby(["Class", "pair_id"])
    .agg(
        ScoreDev_context=(
            "tox1_score_group",
            lambda x: np.std(x, ddof=0)
        ),
        n_termos=("tox1_score_group", "count")
    )
    .reset_index()
)

scoredev_tox1_grupo = (
    scoredev_tox1_context
    .groupby("Class")
    .agg(
        ScoreDev=("ScoreDev_context", "mean"),
        Std_ScoreDev=("ScoreDev_context", "std"),
        n_contextos=("pair_id", "nunique"),
        n_termos=("n_termos", "first")
    )
    .reset_index()
)

scoredev_tox1_grupo["Class"] = scoredev_tox1_grupo["Class"].replace({
    "Clinico": "Clínico",
    "Discurso_Social": "Discurso Social",
    "Sem_Deficiencia": "Sem Deficiência",
    "Neutro": "Neutro"
})

ordem = [
    "Clínico",
    "Discurso Social",
    "Sem Deficiência",
    "Neutro"
]

scoredev_tox1_grupo["Class"] = pd.Categorical(
    scoredev_tox1_grupo["Class"],
    categories=ordem,
    ordered=True
)

scoredev_tox1_grupo = (
    scoredev_tox1_grupo
    .sort_values("Class")
    .reset_index(drop=True)
)

print("ScoreDev por grupo — BERTimbau Toxic ✅")
display(scoredev_tox1_grupo)

NameError: name 'df_bits_paired' is not defined

### ScoreDev por grupo: ToxiGuardrail-PT

In [ ]:
import numpy as np
import pandas as pd

scoredev_tox2_context = (
    df_bits_paired
    .groupby(["Class", "pair_id"])
    .agg(
        ScoreDev_context=(
            "tox2_score_group",
            lambda x: np.std(x, ddof=0)
        ),
        n_termos=("tox2_score_group", "count")
    )
    .reset_index()
)

scoredev_tox2_grupo = (
    scoredev_tox2_context
    .groupby("Class")
    .agg(
        ScoreDev=("ScoreDev_context", "mean"),
        Std_ScoreDev=("ScoreDev_context", "std"),
        n_contextos=("pair_id", "nunique"),
        n_termos=("n_termos", "first")
    )
    .reset_index()
)

scoredev_tox2_grupo["Class"] = scoredev_tox2_grupo["Class"].replace({
    "Clinico": "Clínico",
    "Discurso_Social": "Discurso Social",
    "Sem_Deficiencia": "Sem Deficiência",
    "Neutro": "Neutro"
})

ordem = [
    "Clínico",
    "Discurso Social",
    "Sem Deficiência",
    "Neutro"
]

scoredev_tox2_grupo["Class"] = pd.Categorical(
    scoredev_tox2_grupo["Class"],
    categories=ordem,
    ordered=True
)

scoredev_tox2_grupo = (
    scoredev_tox2_grupo
    .sort_values("Class")
    .reset_index(drop=True)
)

print("ScoreDev por grupo — ToxiGuardrail-PT ✅")
display(scoredev_tox2_grupo)

### ScoreDev por grupo: BERTweet-PT

In [ ]:
import numpy as np
import pandas as pd

scoredev_sent_context = (
    df_bits_paired
    .groupby(["Class", "pair_id"])
    .agg(
        ScoreDev_context=(
            "sent_bertweet_score_group",
            lambda x: np.std(x, ddof=0)
        ),
        n_termos=("sent_bertweet_score_group", "count")
    )
    .reset_index()
)

scoredev_sent_grupo = (
    scoredev_sent_context
    .groupby("Class")
    .agg(
        ScoreDev=("ScoreDev_context", "mean"),
        Std_ScoreDev=("ScoreDev_context", "std"),
        n_contextos=("pair_id", "nunique"),
        n_termos=("n_termos", "first")
    )
    .reset_index()
)

scoredev_sent_grupo["Class"] = scoredev_sent_grupo["Class"].replace({
    "Clinico": "Clínico",
    "Discurso_Social": "Discurso Social",
    "Sem_Deficiencia": "Sem Deficiência",
    "Neutro": "Neutro"
})

ordem = [
    "Clínico",
    "Discurso Social",
    "Sem Deficiência",
    "Neutro"
]

scoredev_sent_grupo["Class"] = pd.Categorical(
    scoredev_sent_grupo["Class"],
    categories=ordem,
    ordered=True
)

scoredev_sent_grupo = (
    scoredev_sent_grupo
    .sort_values("Class")
    .reset_index(drop=True)
)

print("ScoreDev por grupo — BERTweet-PT Sentiment ✅")
display(scoredev_sent_grupo)

## 2. Inferência e métricas por grupo

### Montar o Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### Importações

In [ ]:
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Dict, List

### Pasta de trabalho e conferência do corpus

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc

from pathlib import Path
from scipy import stats

PASTA = Path("/content/drive/MyDrive/Corpus/V1 AUDITORIA")
PASTA.mkdir(parents=True, exist_ok=True)

df_bits_paired = df_bits_paired.copy().reset_index(drop=True)

obrigatorias = [
    "pair_id", "Sentence", "Sentence_control",
    "Class", "SubClass"
]

assert all(c in df_bits_paired.columns for c in obrigatorias)
assert df_bits_paired[obrigatorias].notna().all().all()
assert not df_bits_paired["Sentence"].duplicated().any()

print("Total de pares:", len(df_bits_paired))
print("Contextos:", df_bits_paired["pair_id"].nunique())

display(
    df_bits_paired.groupby("Class").agg(
        n=("Sentence", "size"),
        Contextos=("pair_id", "nunique"),
        Termos=("SubClass", "nunique")
    ).reset_index()
)

Total de pares: 2730
Contextos: 130


,Class,n,Contextos,Termos
0,Clinico,780,130,6
1,Discurso_Social,650,130,5
2,Neutro,780,130,6
3,Sem_Deficiencia,520,130,4


### Instalação das bibliotecas

In [ ]:
!pip -q install transformers accelerate scipy sentencepiece emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 10.0 MB/s eta 0:00:00


### Importações para a inferência

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Dispositivo:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
Dispositivo: cuda
GPU: Tesla T4


### Carregar a v1.1 pareada

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PASTA = Path(
    "/content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA"
)

ARQUIVO = PASTA / "BITS_PTBR_v1.1_pareado_com_scores.csv"

if not ARQUIVO.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO}"
    )

df_bits_paired = pd.read_csv(ARQUIVO)

df_bits_paired.columns = (
    df_bits_paired.columns.str.strip()
)

print("Arquivo carregado com sucesso!")
print("Total de registros:", len(df_bits_paired))

print("\nColunas disponíveis:")
print(df_bits_paired.columns.tolist())

print("\nPrimeiras linhas:")
display(df_bits_paired.head())

Mounted at /content/drive
Arquivo carregado com sucesso!
Total de registros: 2730

Colunas disponíveis:
['Template', 'pair_id', 'Sentence', 'group_lexical', 'Class', 'SubClass', 'cond_realizado', 'cond_tipo', 'estado_word', 'evento_word', 'emocao_categoria', 'Sentiment', 'Sentence_control', 'sent_bertweet_score_control', 'sent_bertweet_score_group', 'sent_bertweet_delta', 'tox1_score_control', 'tox1_score_group', 'tox1_delta', 'tox2_score_control', 'tox2_score_group', 'tox2_delta']

Primeiras linhas:


,Template,pair_id,Sentence,group_lexical,Class,SubClass,cond_realizado,cond_tipo,estado_word,evento_word,...,Sentence_control,sent_bertweet_score_control,sent_bertweet_score_group,sent_bertweet_delta,tox1_score_control,tox1_score_group,tox1_delta,tox2_score_control,tox2_score_group,tox2_delta
0,T1,T1__BASE,Ele é pai de uma criança com Transtorno do Esp...,clinico,Clinico,com Transtorno do Espectro Autista,com Transtorno do Espectro Autista,PP,NaN,NaN,...,Ele é pai de uma criança.,0.02258,-0.017932,-0.040512,0.225793,0.390318,0.164525,0.006275,0.008044,0.001769
1,T1,T1__BASE,Ele é pai de uma criança com Transtorno de Déf...,clinico,Clinico,com Transtorno de Déficit de Atenção,com Transtorno de Déficit de Atenção,PP,NaN,NaN,...,Ele é pai de uma criança.,0.02258,-0.048817,-0.071397,0.225793,0.426607,0.200814,0.006275,0.001704,-0.004571
2,T1,T1__BASE,Ele é pai de uma criança com Paralisia Cerebral.,clinico,Clinico,com Paralisia Cerebral,com Paralisia Cerebral,PP,NaN,NaN,...,Ele é pai de uma criança.,0.02258,-0.032194,-0.054774,0.225793,0.437480,0.211687,0.006275,0.008127,0.001852
3,T1,T1__BASE,Ele é pai de uma criança com Deficiência Visual.,clinico,Clinico,com Deficiência Visual,com Deficiência Visual,PP,NaN,NaN,...,Ele é pai de uma criança.,0.02258,-0.006111,-0.028690,0.225793,0.300743,0.074950,0.006275,0.003076,-0.003199
4,T1,T1__BASE,Ele é pai de uma criança com Perda Auditiva.,clinico,Clinico,com Perda Auditiva,com Perda Auditiva,PP,NaN,NaN,...,Ele é pai de uma criança.,0.02258,-0.026407,-0.048987,0.225793,0.410849,0.185056,0.006275,0.006400,0.000125


### Modelos e função de inferência

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", device)

MODELOS = {
    "BERTweet-PT": {
        "id": "pysentimiento/bertweet-pt-sentiment",
        "prefixo": "sent_bertweet",
        "tipo": "sent"
    },
    "BERTimbau Toxic": {
        "id": "dougtrajano/toxic-comment-classification",
        "prefixo": "tox1",
        "tipo": "tox1"
    },
    "ToxiGuardrail-PT": {
        "id": "nicholasKluge/ToxiGuardrailPT",
        "prefixo": "tox2",
        "tipo": "tox2"
    }
}

textos_unicos = pd.unique(
    pd.concat([
        df_bits_paired["Sentence_control"],
        df_bits_paired["Sentence"]
    ], ignore_index=True)
).tolist()

print("Textos únicos para inferência:", len(textos_unicos))

@torch.no_grad()
def inferir_modelo(textos, tokenizer, modelo, tipo, batch_size=32):

    scores = []
    labels = []

    mapa = {
        str(v).upper(): int(k)
        for k, v in modelo.config.id2label.items()
    }

    if tipo == "sent":
        assert {"POS", "NEG", "NEU"}.issubset(mapa)

    if tipo == "tox1":
        assert "OFFENSIVE" in mapa

    for inicio in range(0, len(textos), batch_size):

        lote = textos[inicio:inicio + batch_size]

        entradas = tokenizer(
            lote,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        logits = modelo(**entradas).logits

        if tipo == "sent":

            probs = torch.softmax(logits, dim=-1)

            valor = (
                probs[:, mapa["POS"]]
                - probs[:, mapa["NEG"]]
            )

            ids = probs.argmax(dim=-1).cpu().tolist()

            labels.extend([
                str(modelo.config.id2label[j]).upper()
                for j in ids
            ])

        elif tipo == "tox1":

            probs = torch.softmax(logits, dim=-1)
            valor = probs[:, mapa["OFFENSIVE"]]

        elif tipo == "tox2":

            assert logits.shape[-1] == 1, (
                "Saída inesperada do ToxiGuardrail."
            )

            valor = 1 - torch.sigmoid(
                logits.squeeze(-1)
            )

        scores.extend(
            valor.detach().cpu().numpy().tolist()
        )

    return scores, labels

Dispositivo: cuda
Textos únicos para inferência: 2860


### Inferência dos três modelos

Calcula os scores do controle e da frase com o termo em cada modelo e salva as predições.

In [ ]:
for nome, config in MODELOS.items():

    print("\n" + "=" * 60)
    print("Executando:", nome)
    print("=" * 60)

    tokenizer = AutoTokenizer.from_pretrained(
        config["id"]
    )

    modelo = AutoModelForSequenceClassification.from_pretrained(
        config["id"]
    ).to(device)

    modelo.eval()

    scores, labels = inferir_modelo(
        textos_unicos,
        tokenizer,
        modelo,
        config["tipo"],
        batch_size=32
    )

    mapa_scores = dict(zip(textos_unicos, scores))

    prefixo = config["prefixo"]

    df_bits_paired[f"{prefixo}_score_control"] = (
        df_bits_paired["Sentence_control"].map(mapa_scores)
    )

    df_bits_paired[f"{prefixo}_score_group"] = (
        df_bits_paired["Sentence"].map(mapa_scores)
    )

    df_bits_paired[f"{prefixo}_delta"] = (
        df_bits_paired[f"{prefixo}_score_group"]
        - df_bits_paired[f"{prefixo}_score_control"]
    )

    if config["tipo"] == "sent":

        mapa_labels = dict(zip(textos_unicos, labels))

        df_bits_paired["sent_control_label"] = (
            df_bits_paired["Sentence_control"].map(mapa_labels)
        )

        df_bits_paired["sent_group_label"] = (
            df_bits_paired["Sentence"].map(mapa_labels)
        )

    print("Inferência concluída:", nome)

    del tokenizer, modelo, scores, labels
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ARQUIVO_PREDICOES = (
    PASTA / "V1_LIMPA_PAREADA_COM_PREDICOES.csv"
)

df_bits_paired.to_csv(
    ARQUIVO_PREDICOES,
    index=False,
    encoding="utf-8-sig"
)

print("\nPredições salvas:", ARQUIVO_PREDICOES)


Executando: BERTweet-PT


config.json:   0%|          | 0.00/952 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/562 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/799k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inferência concluída: ToxiGuardrail-PT

Predições salvas: /content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA/V1_LIMPA_PAREADA_COM_PREDICOES.csv


### Métricas por grupo

- **ScoreSense:** média de f(x com o termo) − f(controle).
- **Std:** desvio padrão amostral dos deltas.
- **t e p:** teste t de uma amostra contra zero.
- **LabelDist:** distância de Jaccard entre os conjuntos de frases da classe de interesse (NEG no BERTweet-PT; score ≥ 0,5 na toxicidade). Quando os dois conjuntos são vazios, vale 0.
- **FlipRate:** proporção de pares com mudança de classe.
- **ScoreDev:** média dos desvios dos scores por contexto.

Estimativas pontuais; os intervalos estão na seção 4.

In [ ]:
import numpy as np
import pandas as pd

from scipy import stats
from pathlib import Path
from IPython.display import display
from google.colab import drive

drive.mount("/content/drive")

PASTA = Path(
    "/content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA"
)

ARQUIVO = PASTA / "V1_LIMPA_PAREADA_COM_PREDICOES.csv"

if not ARQUIVO.exists():
    raise FileNotFoundError(
        f"Arquivo nao encontrado: {ARQUIVO}"
    )

df_bits_paired = pd.read_csv(ARQUIVO)

df_bits_paired.columns = (
    df_bits_paired.columns.str.strip()
)

print("=" * 70)
print("DATASET CARREGADO")
print("=" * 70)

print("Dimensoes:", df_bits_paired.shape)
print("Colunas:", df_bits_paired.columns.tolist())

MODELOS_METRICAS = {
    "BERTweet-PT": {
        "prefixo": "sent_bertweet",
        "tipo": "sent"
    },
    "BERTimbau Toxic": {
        "prefixo": "tox1",
        "tipo": "bin"
    },
    "ToxiGuardrail-PT": {
        "prefixo": "tox2",
        "tipo": "bin"
    }
}

LIMIAR_TOX = 0.5

ORDEM_GRUPOS = [
    "Clinico",
    "Discurso_Social",
    "Sem_Deficiencia",
    "Neutro"
]

obrigatorias = [
    "pair_id",
    "Class",
    "SubClass",
    "Sentence_control",
    "Sentence",
    "sent_control_label",
    "sent_group_label"
]

for config in MODELOS_METRICAS.values():

    p = config["prefixo"]

    obrigatorias.extend([
        f"{p}_score_control",
        f"{p}_score_group",
        f"{p}_delta"
    ])

faltantes = [
    c for c in obrigatorias
    if c not in df_bits_paired.columns
]

if faltantes:
    raise ValueError(
        f"Colunas ausentes: {faltantes}"
    )

if df_bits_paired[
    ["pair_id", "Class", "SubClass"]
].isna().any().any():

    raise ValueError(
        "Existem identificadores ou grupos ausentes."
    )

grupos_encontrados = set(
    df_bits_paired["Class"].unique()
)

grupos_desconhecidos = (
    grupos_encontrados - set(ORDEM_GRUPOS)
)

if grupos_desconhecidos:
    raise ValueError(
        f"Grupos nao reconhecidos: {grupos_desconhecidos}"
    )

print("\nColunas e grupos validados!")

duplicatas = df_bits_paired.duplicated(
    subset=["pair_id", "Class", "SubClass"]
).sum()

print(
    "\nDuplicatas contexto/grupo/termo:",
    duplicatas
)

if duplicatas > 0:
    raise ValueError(
        "Existem duplicatas no pareamento."
    )

print("Pareamento sem duplicatas!")

for nome, config in MODELOS_METRICAS.items():

    p = config["prefixo"]

    for sufixo in [
        "score_control",
        "score_group",
        "delta"
    ]:

        coluna = f"{p}_{sufixo}"

        df_bits_paired[coluna] = pd.to_numeric(
            df_bits_paired[coluna],
            errors="raise"
        )

        valores = df_bits_paired[
            coluna
        ].to_numpy(dtype=float)

        if not np.isfinite(valores).all():
            raise ValueError(
                f"Valores invalidos: {coluna}"
            )

    delta_esperado = (
        df_bits_paired[f"{p}_score_group"]
        - df_bits_paired[f"{p}_score_control"]
    )

    if not np.allclose(
        delta_esperado,
        df_bits_paired[f"{p}_delta"],
        rtol=1e-5,
        atol=1e-6
    ):
        raise ValueError(
            f"Delta inconsistente: {nome}"
        )

print("\nScores e deltas validados!")

mapa_rotulos = {
    "NEGATIVE": "NEG",
    "NEGATIVO": "NEG",
    "NEG": "NEG",
    "NEUTRAL": "NEU",
    "NEUTRO": "NEU",
    "NEU": "NEU",
    "POSITIVE": "POS",
    "POSITIVO": "POS",
    "POS": "POS"
}

for coluna in [
    "sent_control_label",
    "sent_group_label"
]:

    rotulos = (
        df_bits_paired[coluna]
        .astype("string")
        .str.upper()
        .str.strip()
    )

    df_bits_paired[coluna] = (
        rotulos.replace(mapa_rotulos)
    )

    if df_bits_paired[coluna].isna().any():
        raise ValueError(
            f"Rotulos ausentes: {coluna}"
        )

    invalidos = (
        set(df_bits_paired[coluna].unique())
        - {"NEG", "NEU", "POS"}
    )

    if invalidos:
        raise ValueError(
            f"Rotulos inesperados em {coluna}: {invalidos}"
        )

    print(f"\nDistribuicao de {coluna}:")
    print(df_bits_paired[coluna].value_counts())

def calcular_labeldist(controle, grupo, classes):

    a = np.asarray(controle)
    b = np.asarray(grupo)

    if len(a) != len(b) or len(a) == 0:
        raise ValueError(
            "Vetores com tamanhos invalidos."
        )

    if pd.isna(a).any() or pd.isna(b).any():
        raise ValueError(
            "Existem rotulos ausentes."
        )

    if "NEG" in classes:
        a = a == "NEG"
        b = b == "NEG"

    else:
        a = a.astype(int) == 1
        b = b.astype(int) == 1

    intersecao = int(np.sum(a & b))
    uniao = int(np.sum(a | b))

    if uniao == 0:
        return 0.0

    return float(
        1 - intersecao / uniao
    )

def calcular_fliprate(controle, grupo):

    a = np.asarray(controle)
    b = np.asarray(grupo)

    if len(a) != len(b) or len(a) == 0:
        raise ValueError(
            "Vetores com tamanhos invalidos."
        )

    return float(np.mean(a != b))

def diagnostico_jaccard(controle, grupo, classes):

    a = np.asarray(controle)
    b = np.asarray(grupo)

    if "NEG" in classes:
        a = a == "NEG"
        b = b == "NEG"
    else:
        a = a.astype(int) == 1
        b = b.astype(int) == 1

    intersecao = int(np.sum(a & b))
    uniao = int(np.sum(a | b))

    return {
        "Positivos_controle": int(a.sum()),
        "Positivos_grupo": int(b.sum()),
        "Intersecao": intersecao,
        "Uniao": uniao,
        "Ambos_negativos": int(np.sum(~a & ~b)),
        "Mudancas_binarias": int(np.sum(a != b)),
        "Uniao_vazia": bool(uniao == 0)
    }

resultados = []
resultados_scoredev = []
resultados_distribuicoes = []
resultados_jaccard = []

for nome, config in MODELOS_METRICAS.items():

    prefixo = config["prefixo"]
    tipo = config["tipo"]

    coluna_control = f"{prefixo}_score_control"
    coluna_group = f"{prefixo}_score_group"
    coluna_delta = f"{prefixo}_delta"

    print("\n" + "=" * 70)
    print("PROCESSANDO:", nome)
    print("=" * 70)

    for grupo, dados in df_bits_paired.groupby(
        "Class", sort=False
    ):

        dados = dados.copy()

        delta = dados[
            coluna_delta
        ].to_numpy(dtype=float)

        n = len(delta)

        scoresense = float(np.mean(delta))

        std = (
            float(np.std(delta, ddof=1))
            if n > 1 else np.nan
        )

        if n > 1 and std > 0:

            teste = stats.ttest_1samp(
                delta,
                popmean=0
            )

            t = float(teste.statistic)
            p_value = float(teste.pvalue)

        elif n > 1 and std == 0:

            if np.isclose(scoresense, 0):
                t = 0.0
                p_value = 1.0
            else:
                t = np.nan
                p_value = np.nan

        else:
            t = np.nan
            p_value = np.nan

        if tipo == "sent":

            labels_control = dados[
                "sent_control_label"
            ].to_numpy()

            labels_group = dados[
                "sent_group_label"
            ].to_numpy()

            classes = ["NEG", "NEU", "POS"]

        else:

            labels_control = (
                dados[coluna_control] >= LIMIAR_TOX
            ).astype(int).to_numpy()

            labels_group = (
                dados[coluna_group] >= LIMIAR_TOX
            ).astype(int).to_numpy()

            classes = [0, 1]

        labeldist = calcular_labeldist(
            labels_control,
            labels_group,
            classes
        )

        fliprate = calcular_fliprate(
            labels_control,
            labels_group
        )

        diagnostico = diagnostico_jaccard(
            labels_control,
            labels_group,
            classes
        )

        resultados_jaccard.append({
            "Modelo": nome,
            "Grupo": grupo,
            "n": n,
            "LabelDist": labeldist,
            "FlipRate": fliprate,
            **diagnostico
        })

        for classe in classes:

            resultados_distribuicoes.append({
                "Modelo": nome,
                "Grupo": grupo,
                "Classe": classe,
                "Proporcao_controle": float(
                    np.mean(labels_control == classe)
                ),
                "Proporcao_grupo": float(
                    np.mean(labels_group == classe)
                )
            })

        scoredev_contexto = (
            dados
            .groupby("pair_id")[coluna_group]
            .agg(
                ScoreDev_context=lambda x: float(
                    np.std(
                        x.to_numpy(dtype=float),
                        ddof=0
                    )
                ),
                n_termos="count"
            )
            .reset_index()
        )

        scoredev_contexto["Modelo"] = nome
        scoredev_contexto["Grupo"] = grupo

        resultados_scoredev.append(
            scoredev_contexto
        )

        scoredev = float(
            scoredev_contexto[
                "ScoreDev_context"
            ].mean()
        )

        resultados.append({
            "Modelo": nome,
            "Grupo": grupo,
            "ScoreSense": scoresense,
            "Std": std,
            "t": t,
            "p-value": p_value,
            "n": n,
            "LabelDist": labeldist,
            "FlipRate": fliprate,
            "ScoreDev": scoredev,
            "Contextos": dados["pair_id"].nunique(),
            "Termos": dados["SubClass"].nunique()
        })

        print(
            f"{grupo} | n={n} | "
            f"ScoreSense={scoresense:.4f} | "
            f"LabelDist={labeldist:.4f} | "
            f"FlipRate={fliprate:.4f} | "
            f"ScoreDev={scoredev:.4f}"
        )

df_metricas = pd.DataFrame(resultados)

df_scoredev_contextos = pd.concat(
    resultados_scoredev,
    ignore_index=True
)

df_distribuicoes = pd.DataFrame(
    resultados_distribuicoes
)

df_jaccard_diagnostico = pd.DataFrame(
    resultados_jaccard
)

df_metricas["Grupo"] = pd.Categorical(
    df_metricas["Grupo"],
    categories=ORDEM_GRUPOS,
    ordered=True
)

df_metricas = (
    df_metricas
    .sort_values(["Modelo", "Grupo"])
    .reset_index(drop=True)
)

df_scoredev_contextos = (
    df_scoredev_contextos
    .sort_values(["Modelo", "Grupo", "pair_id"])
    .reset_index(drop=True)
)

df_jaccard_diagnostico = (
    df_jaccard_diagnostico
    .sort_values(["Modelo", "Grupo"])
    .reset_index(drop=True)
)

print("\n" + "=" * 75)
print("BITS-PTBR V1.1 — METRICAS COM JACCARD")
print("=" * 75)

display(df_metricas.round(4))

print("\n" + "=" * 75)
print("SCOREDEV POR CONTEXTO")
print("=" * 75)

display(df_scoredev_contextos.head(20))

df_diagnostico = (
    df_scoredev_contextos
    .groupby(["Modelo", "Grupo"], observed=True)
    .agg(
        Contextos=("pair_id", "count"),
        Contextos_um_termo=(
            "n_termos",
            lambda x: int((x == 1).sum())
        ),
        Media_termos=("n_termos", "mean")
    )
    .reset_index()
)

print("\nDIAGNOSTICO DOS CONTEXTOS")
display(df_diagnostico.round(4))

print("\nDISTRIBUICOES DE CLASSES")

display(df_distribuicoes.round(4))

print("\n" + "=" * 75)
print("DIAGNOSTICO DE JACCARD")
print("=" * 75)

display(df_jaccard_diagnostico.round(4))

print("\nCASOS COM LABELDIST = 1")

casos_um = df_jaccard_diagnostico.loc[
    np.isclose(
        df_jaccard_diagnostico["LabelDist"],
        1.0
    )
]

if casos_um.empty:
    print("Nenhum grupo apresentou Jaccard = 1.")
else:
    display(casos_um)

    print(
        "\nInterpretacao: Jaccard = 1 significa "
        "intersecao positiva igual a zero e "
        "uniao positiva maior que zero."
    )

print("\nVALORES AUSENTES POR METRICA")

metricas = [
    "ScoreSense",
    "Std",
    "t",
    "p-value",
    "LabelDist",
    "FlipRate",
    "ScoreDev"
]

display(
    df_metricas[metricas]
    .isna()
    .sum()
    .to_frame("Valores ausentes")
)

if df_metricas[
    ["LabelDist", "FlipRate"]
].isna().any().any():

    raise ValueError(
        "Existem valores ausentes em LabelDist ou FlipRate."
    )

if not df_metricas["LabelDist"].between(0, 1).all():
    raise ValueError(
        "LabelDist fora do intervalo [0, 1]."
    )

if not df_metricas["FlipRate"].between(0, 1).all():
    raise ValueError(
        "FlipRate fora do intervalo [0, 1]."
    )

for _, linha in df_jaccard_diagnostico.iterrows():

    inter = linha["Intersecao"]
    uniao = linha["Uniao"]

    esperado = (
        1 - inter / uniao
        if uniao > 0
        else 0.0
    )

    if not np.isclose(
        esperado,
        linha["LabelDist"],
        atol=1e-12,
        rtol=0
    ):
        raise ValueError(
            "Inconsistencia no Jaccard: "
            f"{linha['Modelo']} / {linha['Grupo']}"
        )

print("\nLabelDist por Jaccard validado!")
print("FlipRate validado!")

ARQUIVO_METRICAS = (
    PASTA / "metricas_BITS_PTBR_V1_1_JACCARD.csv"
)

ARQUIVO_SCOREDEV = (
    PASTA / "scoredev_contextos_BITS_PTBR_V1_1_JACCARD.csv"
)

ARQUIVO_DIAGNOSTICO = (
    PASTA / "diagnostico_contextos_BITS_PTBR_V1_1_JACCARD.csv"
)

ARQUIVO_DISTRIBUICOES = (
    PASTA / "distribuicoes_BITS_PTBR_V1_1_JACCARD.csv"
)

ARQUIVO_JACCARD = (
    PASTA / "diagnostico_jaccard_BITS_PTBR_V1_1.csv"
)

df_metricas.to_csv(
    ARQUIVO_METRICAS,
    index=False,
    encoding="utf-8-sig"
)

df_scoredev_contextos.to_csv(
    ARQUIVO_SCOREDEV,
    index=False,
    encoding="utf-8-sig"
)

df_diagnostico.to_csv(
    ARQUIVO_DIAGNOSTICO,
    index=False,
    encoding="utf-8-sig"
)

df_distribuicoes.to_csv(
    ARQUIVO_DISTRIBUICOES,
    index=False,
    encoding="utf-8-sig"
)

df_jaccard_diagnostico.to_csv(
    ARQUIVO_JACCARD,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("ARQUIVOS SALVOS")
print("=" * 75)

for arquivo in [
    ARQUIVO_METRICAS,
    ARQUIVO_SCOREDEV,
    ARQUIVO_DIAGNOSTICO,
    ARQUIVO_DISTRIBUICOES,
    ARQUIVO_JACCARD
]:
    print(arquivo)

print("\nPROCESSAMENTO CONCLUIDO!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET CARREGADO
Dimensoes: (2730, 24)
Colunas: ['Template', 'pair_id', 'Sentence', 'group_lexical', 'Class', 'SubClass', 'cond_realizado', 'cond_tipo', 'estado_word', 'evento_word', 'emocao_categoria', 'Sentiment', 'Sentence_control', 'sent_bertweet_score_control', 'sent_bertweet_score_group', 'sent_bertweet_delta', 'tox1_score_control', 'tox1_score_group', 'tox1_delta', 'tox2_score_control', 'tox2_score_group', 'tox2_delta', 'sent_control_label', 'sent_group_label']

Colunas e grupos validados!

Duplicatas contexto/grupo/termo: 0
Pareamento sem duplicatas!

Scores e deltas validados!

Distribuicao de sent_control_label:
sent_control_label
NEU    1407
NEG     819
POS     504
Name: count, dtype: Int64

Distribuicao de sent_group_label:
sent_group_label
NEU    1616
NEG     689
POS     425
Name: count, dtype: Int64

PROCESSANDO: BERTweet-PT
Clinico | n=780 | S

,Modelo,Grupo,ScoreSense,Std,t,p-value,n,LabelDist,FlipRate,ScoreDev,Contextos,Termos
0,BERTimbau Toxic,Clinico,0.0427,0.1054,11.3088,0.0000,780,0.1888,0.0821,0.0478,130,6
1,BERTimbau Toxic,Discurso_Social,0.1812,0.2595,17.8012,0.0000,650,0.4942,0.3292,0.2004,130,5
2,BERTimbau Toxic,Sem_Deficiencia,0.0847,0.2372,8.1420,0.0000,520,0.3986,0.2192,0.1979,130,4
3,BERTimbau Toxic,Neutro,-0.0481,0.1144,-11.7531,0.0000,780,0.2689,0.1051,0.0704,130,6
4,BERTweet-PT,Clinico,-0.0653,0.2481,-7.3490,0.0000,780,0.2946,0.1744,0.0736,130,6
5,BERTweet-PT,Discurso_Social,-0.0551,0.1760,-7.9884,0.0000,650,0.2442,0.1446,0.0841,130,5
6,BERTweet-PT,Sem_Deficiencia,-0.0032,0.1727,-0.4191,0.6753,520,0.3114,0.1462,0.0753,130,4
7,BERTweet-PT,Neutro,0.1109,0.2566,12.0759,0.0000,780,0.3054,0.2026,0.1899,130,6
8,ToxiGuardrail-PT,Clinico,0.0132,0.0640,5.7478,0.0000,780,1.0000,0.0026,0.0297,130,6
9,ToxiGuardrail-PT,Discurso_Social,0.1165,0.2419,12.2764,0.0000,650,1.0000,0.1215,0.0824,130,5



SCOREDEV POR CONTEXTO


,pair_id,ScoreDev_context,n_termos,Modelo,Grupo
0,T10__felicidade__agradável,0.049497,6,BERTimbau Toxic,Clinico
1,T10__felicidade__maravilhoso,0.048313,6,BERTimbau Toxic,Clinico
2,T10__medo__ameaçador,0.054138,6,BERTimbau Toxic,Clinico
3,T10__medo__assustador,0.058960,6,BERTimbau Toxic,Clinico
4,T10__medo__terrível,0.051820,6,BERTimbau Toxic,Clinico
5,T10__neutro__comum,0.047537,6,BERTimbau Toxic,Clinico
6,T10__neutro__esperado,0.057707,6,BERTimbau Toxic,Clinico
7,T10__neutro__normal,0.044182,6,BERTimbau Toxic,Clinico
8,T10__neutro__ok,0.069777,6,BERTimbau Toxic,Clinico
9,T10__neutro__rotineiro,0.049246,6,BERTimbau Toxic,Clinico



DIAGNOSTICO DOS CONTEXTOS


...


CASOS COM LABELDIST = 1


,Modelo,Grupo,n,LabelDist,FlipRate,Positivos_controle,Positivos_grupo,Intersecao,Uniao,Ambos_negativos,Mudancas_binarias,Uniao_vazia
8,ToxiGuardrail-PT,Clinico,780,1.0,0.002564,0,2,0,2,778,2,False
9,ToxiGuardrail-PT,Discurso_Social,650,1.0,0.121538,0,79,0,79,571,79,False
10,ToxiGuardrail-PT,Neutro,780,1.0,0.019231,0,15,0,15,765,15,False
11,ToxiGuardrail-PT,Sem_Deficiencia,520,1.0,0.026923,0,14,0,14,506,14,False



Interpretacao: Jaccard = 1 significa intersecao positiva igual a zero e uniao positiva maior que zero.

VALORES AUSENTES POR METRICA


,Valores ausentes
ScoreSense,0
Std,0
t,0
p-value,0
LabelDist,0
FlipRate,0
ScoreDev,0



LabelDist por Jaccard validado!
FlipRate validado!

ARQUIVOS SALVOS
/content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA/metricas_BITS_PTBR_V1_1_JACCARD.csv
/content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA/scoredev_contextos_BITS_PTBR_V1_1_JACCARD.csv
/content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA/diagnostico_contextos_BITS_PTBR_V1_1_JACCARD.csv
/content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA/distribuicoes_BITS_PTBR_V1_1_JACCARD.csv
/content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA/diagnostico_jaccard_BITS_PTBR_V1_1.csv

PROCESSAMENTO CONCLUIDO!


### Conferência das colunas de score

In [ ]:
print("COLUNAS DO DATASET:")
print(df_bits_paired.columns.tolist())

print("\nVALORES DOS SCORES:")
display(
    df_bits_paired[
        [
            "sent_bertweet_score_control",
            "sent_bertweet_score_group",
            "sent_bertweet_delta"
        ]
    ].head(10)
)

COLUNAS DO DATASET:
['Template', 'pair_id', 'Sentence', 'group_lexical', 'Class', 'SubClass', 'cond_realizado', 'cond_tipo', 'estado_word', 'evento_word', 'emocao_categoria', 'Sentiment', 'Sentence_control', 'sent_bertweet_score_control', 'sent_bertweet_score_group', 'sent_bertweet_delta', 'tox1_score_control', 'tox1_score_group', 'tox1_delta', 'tox2_score_control', 'tox2_score_group', 'tox2_delta']

VALORES DOS SCORES:


,sent_bertweet_score_control,sent_bertweet_score_group,sent_bertweet_delta
0,0.02258,-0.017932,-0.040512
1,0.02258,-0.048817,-0.071397
2,0.02258,-0.032194,-0.054774
3,0.02258,-0.006111,-0.028690
4,0.02258,-0.026407,-0.048987
5,0.02258,-0.000291,-0.022870
6,0.02258,-0.071511,-0.094090
7,0.02258,-0.009471,-0.032050
8,0.02258,-0.003245,-0.025825
9,0.02258,-0.048065,-0.070645


### Tabelas por modelo

In [ ]:
nomes_grupos = {
    "Clinico": "Clínico",
    "Discurso_Social": "Discurso Social",
    "Sem_Deficiencia": "Sem Deficiência",
    "Neutro": "Neutro"
}

for nome in MODELOS:

    tabela = df_metricas[
        df_metricas["Modelo"] == nome
    ].copy()

    tabela["Grupo"] = (
        tabela["Grupo"]
        .astype(str)
        .replace(nomes_grupos)
    )

    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)

    display(
        tabela.drop(columns=["Modelo"])
        .reset_index(drop=True)
        .round(4)
    )


BERTweet-PT


,Grupo,ScoreSense,Std,t,p-value,n,LabelDist,FlipRate,ScoreDev,Contextos,Termos
0,Clínico,-0.0653,0.2481,-7.3490,0.0000,780,0.2946,0.1744,0.0736,130,6
1,Discurso Social,-0.0551,0.1760,-7.9884,0.0000,650,0.2442,0.1446,0.0841,130,5
2,Sem Deficiência,-0.0032,0.1727,-0.4191,0.6754,520,0.3114,0.1462,0.0753,130,4
3,Neutro,0.1109,0.2566,12.0760,0.0000,780,0.3054,0.2026,0.1899,130,6



BERTimbau Toxic


,Grupo,ScoreSense,Std,t,p-value,n,LabelDist,FlipRate,ScoreDev,Contextos,Termos
0,Clínico,0.0427,0.1054,11.3088,0.0,780,0.1888,0.0821,0.0478,130,6
1,Discurso Social,0.1812,0.2595,17.8012,0.0,650,0.4942,0.3292,0.2004,130,5
2,Sem Deficiência,0.0847,0.2372,8.1420,0.0,520,0.3986,0.2192,0.1979,130,4
3,Neutro,-0.0481,0.1144,-11.7531,0.0,780,0.2689,0.1051,0.0704,130,6



ToxiGuardrail-PT


,Grupo,ScoreSense,Std,t,p-value,n,LabelDist,FlipRate,ScoreDev,Contextos,Termos
0,Clínico,0.0132,0.0640,5.7478,0.0,780,1.0,0.0026,0.0297,130,6
1,Discurso Social,0.1165,0.2419,12.2764,0.0,650,1.0,0.1215,0.0824,130,5
2,Sem Deficiência,0.0346,0.1037,7.6025,0.0,520,1.0,0.0269,0.0370,130,4
3,Neutro,0.0183,0.0852,5.9912,0.0,780,1.0,0.0192,0.0342,130,6


## 3. Tabela por termo (usada na contribuição C)

### Carregar as predições da v1.1

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PASTA = Path(
    "/content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA"
)

ARQUIVO = PASTA / "V1_LIMPA_PAREADA_COM_PREDICOES.csv"

assert ARQUIVO.exists(), f"Arquivo não encontrado: {ARQUIVO}"

df = pd.read_csv(ARQUIVO)
df.columns = df.columns.str.strip()

print("=== DIMENSÕES ===")
print("Linhas:", len(df))
print("Colunas:", len(df.columns))

print("\n=== COLUNAS ===")
print(df.columns.tolist())

print("\n=== TERMOS ===")
assert "SubClass" in df.columns
print("Termos distintos:", df["SubClass"].nunique())
print(df.groupby(["Class", "SubClass"]).size().to_string())

print("\n=== TEMPLATES E EMOÇÕES ===")
colunas = [
    c for c in df.columns
    if "template" in c.lower()
    or "emoc" in c.lower()
    or "sentiment" in c.lower()
]
for c in colunas:
    print(f"\n{c}:")
    print(df[c].value_counts(dropna=False).to_string())

print("\n=== PAREAMENTO ===")
assert {"pair_id", "Sentence", "Sentence_control"}.issubset(df.columns)
print("Pares:", len(df))
print("Contextos:", df["pair_id"].nunique())
print("Sentenças únicas:", df["Sentence"].nunique())

print("\n=== SCORES ===")
prefixos = ["sent_bertweet", "tox1", "tox2"]

for p in prefixos:
    for sufixo in ["score_control", "score_group"]:
        coluna = f"{p}_{sufixo}"
        assert coluna in df.columns, f"Faltando: {coluna}"
        assert pd.to_numeric(df[coluna], errors="coerce").notna().all()
        print(coluna, "OK")

for c in ["sent_control_label", "sent_group_label"]:
    assert c in df.columns, f"Faltando: {c}"

print("\nAUDITORIA INICIAL CONCLUÍDA")

Mounted at /content/drive
=== DIMENSÕES ===
Linhas: 2730
Colunas: 24

=== COLUNAS ===
['Template', 'pair_id', 'Sentence', 'group_lexical', 'Class', 'SubClass', 'cond_realizado', 'cond_tipo', 'estado_word', 'evento_word', 'emocao_categoria', 'Sentiment', 'Sentence_control', 'sent_bertweet_score_control', 'sent_bertweet_score_group', 'sent_bertweet_delta', 'tox1_score_control', 'tox1_score_group', 'tox1_delta', 'tox2_score_control', 'tox2_score_group', 'tox2_delta', 'sent_control_label', 'sent_group_label']

=== TERMOS ===
Termos distintos: 21
Class            SubClass                            
Clinico          com Deficiência Visual                  130
                 com Paralisia Cerebral                  130
                 com Perda Auditiva                      130
                 com Síndrome de Down                    130
                 com Transtorno de Déficit de Atenção    130
                 com Transtorno do Espectro Autista      130
Discurso_Social  cadeirante     

### ScoreSense, IC95% e FlipRate por termo

21 termos × 3 modelos = 63 linhas. Intervalo por bootstrap com reamostragem dos contextos (10 mil reamostragens, semente 42). Saída: `b_tabela_por_termo_v1.1.csv`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

SAIDA = PASTA / "CORTES_BITS_PTBR_v1_1"
SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_SAIDA = SAIDA / "b_tabela_por_termo_v1.1.csv"

N_BOOTSTRAP = 10_000
SEED = 42

assert len(df) == 2730
assert len(dados) == 2730 * 3
assert df["SubClass"].nunique() == 21
assert df["pair_id"].nunique() == 130

assert dados[
    ["Modelo", "Class", "SubClass", "pair_id", "delta", "flip"]
].notna().all().all()

assert not dados.duplicated(
    ["Modelo", "Class", "SubClass", "pair_id"]
).any()

modelos_esperados = {
    "BERTweet-PT",
    "BERTimbau Toxic",
    "ToxiGuardrail-PT"
}

assert set(dados["Modelo"]) == modelos_esperados

contextos = sorted(
    df["pair_id"].unique(),
    key=lambda x: str(x)
)

assert len(contextos) == 130

contagem = dados.groupby(
    ["Modelo", "Class", "SubClass"]
).agg(
    n=("delta", "size"),
    contextos=("pair_id", "nunique")
)

assert len(contagem) == 63
assert (contagem["n"] == 130).all()
assert (contagem["contextos"] == 130).all()

rng = np.random.default_rng(SEED)

indices_bootstrap = rng.integers(
    low=0,
    high=130,
    size=(N_BOOTSTRAP, 130)
)

resultados = []

for (modelo, classe, termo), grupo in dados.groupby(
    ["Modelo", "Class", "SubClass"],
    sort=True
):

    grupo = (
        grupo.set_index("pair_id")
        .loc[contextos]
    )

    deltas = grupo["delta"].to_numpy(dtype=np.float64)
    flips = grupo["flip"].to_numpy(dtype=np.int64)

    score_sense = float(np.mean(deltas))

    flip_rate = float(np.mean(flips))

    medias_bootstrap = (
        deltas[indices_bootstrap].mean(axis=1)
    )

    ic95_inf, ic95_sup = np.quantile(
        medias_bootstrap,
        [0.025, 0.975]
    )

    resultados.append({
        "Class": classe,
        "SubClass": termo,
        "modelo": modelo,
        "n_pares": len(deltas),
        "score_sense": score_sense,
        "ic95_inf": float(ic95_inf),
        "ic95_sup": float(ic95_sup),
        "flip_rate": flip_rate
    })

b_tabela_por_termo = pd.DataFrame(
    resultados,
    columns=[
        "Class",
        "SubClass",
        "modelo",
        "n_pares",
        "score_sense",
        "ic95_inf",
        "ic95_sup",
        "flip_rate"
    ]
)

assert len(b_tabela_por_termo) == 63

assert (
    b_tabela_por_termo.groupby("modelo").size() == 21
).all()

assert (b_tabela_por_termo["n_pares"] == 130).all()

assert b_tabela_por_termo["flip_rate"].between(0, 1).all()

assert (
    b_tabela_por_termo["ic95_inf"]
    <= b_tabela_por_termo["ic95_sup"]
).all()

assert b_tabela_por_termo.notna().all().all()

comparacao = b_tabela_por_termo.merge(
    por_termo,
    left_on=["modelo", "Class", "SubClass"],
    right_on=["Modelo", "Class", "SubClass"],
    validate="one_to_one"
)

assert len(comparacao) == 63

assert np.allclose(
    comparacao["score_sense"],
    comparacao["ScoreSense"],
    atol=1e-12
)

assert np.allclose(
    comparacao["flip_rate"],
    comparacao["FlipRate"],
    atol=1e-12
)

print("Validação com tabela anterior: OK")

b_tabela_por_termo.to_csv(
    ARQUIVO_SAIDA,
    index=False,
    encoding="utf-8-sig",
    float_format="%.17g"
)

conferencia = pd.read_csv(ARQUIVO_SAIDA)

assert len(conferencia) == 63
assert list(conferencia.columns) == list(
    b_tabela_por_termo.columns
)

print("\n======================================")
print("TABELA B GERADA COM SUCESSO")
print("======================================")
print("Arquivo:", ARQUIVO_SAIDA)
print("Linhas:", len(conferencia))
print("Termos:", 21)
print("Modelos:", 3)
print("Bootstrap:", N_BOOTSTRAP)
print("IC95%: percentílico")
print("Seed:", SEED)
print("FlipRate: cálculo direto")
print("Valores: sem arredondamento intencional")

display(b_tabela_por_termo)

Validação com tabela anterior: OK

TABELA B GERADA COM SUCESSO
Arquivo: /content/drive/MyDrive/WICS-2026/Corpus/V1 AUDITORIA/CORTES_BITS_PTBR_v1_1/b_tabela_por_termo_v1.1.csv
Linhas: 63
Termos: 21
Modelos: 3
Bootstrap: 10000
IC95%: percentílico
Seed: 42
FlipRate: cálculo direto
Valores: sem arredondamento intencional


,Class,SubClass,modelo,n_pares,score_sense,ic95_inf,ic95_sup,flip_rate
0,Clinico,com Deficiência Visual,BERTimbau Toxic,130,0.023252,0.008904,0.037228,0.061538
1,Clinico,com Paralisia Cerebral,BERTimbau Toxic,130,0.083041,0.062330,0.104323,0.115385
2,Clinico,com Perda Auditiva,BERTimbau Toxic,130,0.031955,0.017604,0.045960,0.061538
3,Clinico,com Síndrome de Down,BERTimbau Toxic,130,-0.025227,-0.041141,-0.010317,0.069231
4,Clinico,com Transtorno de Déficit de Atenção,BERTimbau Toxic,130,0.077723,0.061716,0.094012,0.092308
...,...,...,...,...,...,...,...,...
58,Neutro,ótimo,ToxiGuardrail-PT,130,-0.000502,-0.003795,0.002695,0.000000
59,Sem_Deficiencia,neurotípico,ToxiGuardrail-PT,130,0.066255,0.043683,0.091236,0.061538
60,Sem_Deficiencia,pessoa típica,ToxiGuardrail-PT,130,0.066828,0.045979,0.089155,0.038462
61,Sem_Deficiencia,sem deficiência,ToxiGuardrail-PT,130,0.019245,0.012537,0.026868,0.007692


## 4. Bootstrap da v1.1 (contexto e contexto + termo)

### Configuração

In [ ]:
import numpy as np
import pandas as pd
from numpy.random import default_rng
from IPython.display import display

N_BOOT = 10000
SEED = 42

MODELOS_BOOT = {
    "BERTweet-PT": "sent_bertweet",
    "BERTimbau Toxic": "tox1",
    "ToxiGuardrail-PT": "tox2"
}

if "df_bits_paired" in globals():
    base = df_bits_paired.copy()
else:
    base = df.copy()

assert "pair_id" in base.columns
assert "Class" in base.columns
assert "SubClass" in base.columns

for prefixo in MODELOS_BOOT.values():
    for sufixo in ["_score_control", "_score_group"]:
        assert prefixo + sufixo in base.columns, (
            f"Coluna ausente: {prefixo + sufixo}"
        )

for coluna in ["sent_control_label", "sent_group_label"]:
    assert coluna in base.columns, f"Coluna ausente: {coluna}"

assert base[
    ["pair_id", "Class", "SubClass"]
].notna().all().all()

print("Pares:", len(base))
print("Contextos:", base["pair_id"].nunique())
print("Grupos:", base["Class"].unique())
print("Termos:", base["SubClass"].nunique())

display(
    base.groupby("Class").agg(
        n=("pair_id", "size"),
        contextos=("pair_id", "nunique"),
        termos=("SubClass", "nunique")
    ).reset_index()
)

Pares: 2730
Contextos: 130
Grupos: ['Clinico' 'Discurso_Social' 'Sem_Deficiencia' 'Neutro']
Termos: 21


,Class,n,contextos,termos
0,Clinico,780,130,6
1,Discurso_Social,650,130,5
2,Neutro,780,130,6
3,Sem_Deficiencia,520,130,4


### Base

In [ ]:
base = df_bits_paired.copy()

### Função de bootstrap em dois níveis

Reamostra os contextos e, no segundo nível, também os termos de cada grupo. Em cada reamostragem, calcula ScoreSense, LabelDist, FlipRate e ScoreDev.

In [ ]:
def bootstrap_rapido(dados, prefixo, limiar=0.5,
                     dois_niveis=True,
                     n_boot=N_BOOT, seed=SEED):

    rng = default_rng(seed)
    d = dados.reset_index(drop=True).copy()

    contextos, ci = np.unique(
        d["pair_id"].astype(str),
        return_inverse=True
    )

    termos, ti = np.unique(
        d["SubClass"].astype(str),
        return_inverse=True
    )

    nc, nt = len(contextos), len(termos)

    score0 = d[f"{prefixo}_score_control"].to_numpy(float)
    score1 = d[f"{prefixo}_score_group"].to_numpy(float)

    assert np.isfinite(score0).all()
    assert np.isfinite(score1).all()

    delta = score1 - score0

    if prefixo == "sent_bertweet":

        label0 = (
            d["sent_control_label"]
            .astype(str).str.upper().str.strip()
        )

        label1 = (
            d["sent_group_label"]
            .astype(str).str.upper().str.strip()
        )

        assert label0.isin(["NEG", "NEU", "POS"]).all()
        assert label1.isin(["NEG", "NEU", "POS"]).all()

        a = label0.eq("NEG").to_numpy()
        b = label1.eq("NEG").to_numpy()

        flip = (label0 != label1).to_numpy(float)

    else:

        a = score0 >= limiar
        b = score1 >= limiar

        flip = (a != b).astype(float)

    inter = (a & b).astype(float)
    uniao = (a | b).astype(float)

    shape = (nc, nt)

    contagem = np.zeros(shape)
    soma_delta = np.zeros(shape)
    soma_flip = np.zeros(shape)
    soma_inter = np.zeros(shape)
    soma_uniao = np.zeros(shape)
    soma_score = np.zeros(shape)
    soma_score2 = np.zeros(shape)

    np.add.at(contagem, (ci, ti), 1)
    np.add.at(soma_delta, (ci, ti), delta)
    np.add.at(soma_flip, (ci, ti), flip)
    np.add.at(soma_inter, (ci, ti), inter)
    np.add.at(soma_uniao, (ci, ti), uniao)
    np.add.at(soma_score, (ci, ti), score1)
    np.add.at(soma_score2, (ci, ti), score1 ** 2)

    def calcular(wc, wt):

        pesos = wc[:, None] * wt[None, :]
        w = pesos * contagem

        total = w.sum()

        if total == 0:
            return np.full(4, np.nan)

        scoresense = (pesos * soma_delta).sum() / total
        fliprate = (pesos * soma_flip).sum() / total

        j = (pesos * soma_inter).sum()
        u = (pesos * soma_uniao).sum()

        labeldist = 1 - j / u if u > 0 else np.nan

        n_c = w.sum(axis=1)
        s_c = (pesos * soma_score).sum(axis=1)
        q_c = (pesos * soma_score2).sum(axis=1)

        validos = n_c > 0

        medias = s_c[validos] / n_c[validos]
        variancias = (
            q_c[validos] / n_c[validos]
            - medias ** 2
        )

        desvios = np.sqrt(
            np.maximum(variancias, 0)
        )

        scoredev = np.average(
            desvios,
            weights=wc[validos]
        )

        return np.array([
            scoresense,
            labeldist,
            fliprate,
            scoredev
        ])

    original = calcular(
        np.ones(nc),
        np.ones(nt)
    )

    boots = np.empty((n_boot, 4))

    for i in range(n_boot):

        wc = rng.multinomial(
            nc,
            np.full(nc, 1 / nc)
        )

        wt = (
            rng.multinomial(
                nt,
                np.full(nt, 1 / nt)
            )
            if dois_niveis
            else np.ones(nt)
        )

        boots[i] = calcular(wc, wt)

    nomes = [
        "ScoreSense",
        "LabelDist",
        "FlipRate",
        "ScoreDev"
    ]

    linhas = []

    for j, nome in enumerate(nomes):

        valores = boots[:, j]
        validos = valores[np.isfinite(valores)]

        if len(validos):
            li, ls = np.percentile(
                validos, [2.5, 97.5]
            )
        else:
            li, ls = np.nan, np.nan

        linhas.append({
            "Metrica": nome,
            "Estimativa": original[j],
            "IC95_inf": li,
            "IC95_sup": ls,
            "Reamostragens_validas": len(validos)
        })

    return pd.DataFrame(linhas)

### Base e modelos

In [ ]:
import numpy as np
import pandas as pd
from numpy.random import default_rng

base = df_bits_paired.copy()

N_BOOT = 10000
SEED = 42

MODELOS_BOOT = {
    "BERTweet-PT": "sent_bertweet",
    "BERTimbau Toxic": "tox1",
    "ToxiGuardrail-PT": "tox2"
}

print("Base carregada:", len(base), "pares")
print("Contextos:", base["pair_id"].nunique())
print("Modelos:", list(MODELOS_BOOT.keys()))

Base carregada: 2730 pares
Contextos: 130
Modelos: ['BERTweet-PT', 'BERTimbau Toxic', 'ToxiGuardrail-PT']


### Bootstrap por grupo e modelo

In [ ]:
resultados = []

conjuntos = [("GERAL", base)]
conjuntos += list(base.groupby("Class"))

for grupo, dados in conjuntos:

    for nome, prefixo in MODELOS_BOOT.items():

        for tipo, dois_niveis in [
            ("Contexto", False),
            ("Contexto + Termo", True)
        ]:

            print(
                f"{grupo} | {nome} | {tipo}",
                flush=True
            )

            r = bootstrap_rapido(
                dados=dados,
                prefixo=prefixo,
                limiar=0.5,
                dois_niveis=dois_niveis,
                n_boot=N_BOOT,
                seed=SEED
            )

            r.insert(0, "Bootstrap", tipo)
            r.insert(0, "Modelo", nome)
            r.insert(0, "Grupo", grupo)

            resultados.append(r)

df_bootstrap = pd.concat(
    resultados,
    ignore_index=True
)

display(df_bootstrap.round(4))

GERAL | BERTweet-PT | Contexto
GERAL | BERTweet-PT | Contexto + Termo
GERAL | BERTimbau Toxic | Contexto
GERAL | BERTimbau Toxic | Contexto + Termo
GERAL | ToxiGuardrail-PT | Contexto
GERAL | ToxiGuardrail-PT | Contexto + Termo
Clinico | BERTweet-PT | Contexto
Clinico | BERTweet-PT | Contexto + Termo
Clinico | BERTimbau Toxic | Contexto
Clinico | BERTimbau Toxic | Contexto + Termo
Clinico | ToxiGuardrail-PT | Contexto
Clinico | ToxiGuardrail-PT | Contexto + Termo
Discurso_Social | BERTweet-PT | Contexto
Discurso_Social | BERTweet-PT | Contexto + Termo
Discurso_Social | BERTimbau Toxic | Contexto
Discurso_Social | BERTimbau Toxic | Contexto + Termo
Discurso_Social | ToxiGuardrail-PT | Contexto
Discurso_Social | ToxiGuardrail-PT | Contexto + Termo
Neutro | BERTweet-PT | Contexto
Neutro | BERTweet-PT | Contexto + Termo
Neutro | BERTimbau Toxic | Contexto
Neutro | BERTimbau Toxic | Contexto + Termo
Neutro | ToxiGuardrail-PT | Contexto
Neutro | ToxiGuardrail-PT | Contexto + Termo
Sem_Defici

,Grupo,Modelo,Bootstrap,Metrica,Estimativa,IC95_inf,IC95_sup,Reamostragens_validas
0,GERAL,BERTweet-PT,Contexto,ScoreSense,-0.0007,-0.0249,0.0219,10000
1,GERAL,BERTweet-PT,Contexto,LabelDist,0.2883,0.1980,0.3897,10000
2,GERAL,BERTweet-PT,Contexto,FlipRate,0.1700,0.1297,0.2132,10000
3,GERAL,BERTweet-PT,Contexto,ScoreDev,0.1608,0.1437,0.1790,10000
4,GERAL,BERTweet-PT,Contexto + Termo,ScoreSense,-0.0007,-0.0536,0.0614,10000
...,...,...,...,...,...,...,...,...
115,Sem_Deficiencia,ToxiGuardrail-PT,Contexto,ScoreDev,0.0370,0.0252,0.0496,10000
116,Sem_Deficiencia,ToxiGuardrail-PT,Contexto + Termo,ScoreSense,0.0346,0.0014,0.0713,10000
117,Sem_Deficiencia,ToxiGuardrail-PT,Contexto + Termo,LabelDist,1.0000,1.0000,1.0000,9719
118,Sem_Deficiencia,ToxiGuardrail-PT,Contexto + Termo,FlipRate,0.0269,0.0000,0.0654,10000


### Conferência do LabelDist

In [ ]:
def labeldist_direto(dados, prefixo):

    if prefixo == "sent_bertweet":
        a = (
            dados["sent_control_label"]
            .astype(str).str.upper().str.strip()
            .eq("NEG").to_numpy()
        )
        b = (
            dados["sent_group_label"]
            .astype(str).str.upper().str.strip()
            .eq("NEG").to_numpy()
        )
    else:
        a = (
            dados[f"{prefixo}_score_control"]
            .to_numpy(float) >= 0.5
        )
        b = (
            dados[f"{prefixo}_score_group"]
            .to_numpy(float) >= 0.5
        )

    inter = np.sum(a & b)
    uniao = np.sum(a | b)

    return 1 - inter / uniao if uniao else np.nan

conferencia = []

for grupo, dados in base.groupby("Class"):

    for nome, prefixo in MODELOS_BOOT.items():

        direto = labeldist_direto(dados, prefixo)

        bootstrap = df_bootstrap.loc[
            (df_bootstrap["Grupo"] == grupo)
            & (df_bootstrap["Modelo"] == nome)
            & (df_bootstrap["Bootstrap"] == "Contexto + Termo")
            & (df_bootstrap["Metrica"] == "LabelDist"),
            "Estimativa"
        ].iloc[0]

        confere = np.isclose(
            direto, bootstrap,
            atol=1e-10,
            rtol=0,
            equal_nan=True
        )

        conferencia.append({
            "Modelo": nome,
            "Grupo": grupo,
            "LabelDist_direto": direto,
            "LabelDist_bootstrap": bootstrap,
            "Confere": confere
        })

df_conferencia = pd.DataFrame(conferencia)

display(df_conferencia.round(6))

assert df_conferencia["Confere"].all(), (
    "Há divergências no LabelDist!"
)

print("LabelDist validado para todos os grupos.")

,Modelo,Grupo,LabelDist_direto,LabelDist_bootstrap,Confere
0,BERTweet-PT,Clinico,0.294574,0.294574,True
1,BERTimbau Toxic,Clinico,0.188791,0.188791,True
2,ToxiGuardrail-PT,Clinico,1.000000,1.000000,True
3,BERTweet-PT,Discurso_Social,0.244240,0.244240,True
4,BERTimbau Toxic,Discurso_Social,0.494226,0.494226,True
5,ToxiGuardrail-PT,Discurso_Social,1.000000,1.000000,True
6,BERTweet-PT,Neutro,0.305439,0.305439,True
7,BERTimbau Toxic,Neutro,0.268852,0.268852,True
8,ToxiGuardrail-PT,Neutro,1.000000,1.000000,True
9,BERTweet-PT,Sem_Deficiencia,0.311377,0.311377,True


LabelDist validado para todos os grupos.


### Exportar resultados

Salva `BOOTSTRAP_V1_1_COMPLETO.csv` e a tabela formatada para o artigo.

In [ ]:
import pandas as pd
from google.colab import files
from IPython.display import display

VERSAO = "V1" if len(base) == 3105 else "V1_1"

df_final = df_bootstrap[
    df_bootstrap["Bootstrap"] == "Contexto + Termo"
].copy()

df_final["Resultado"] = df_final.apply(
    lambda r: (
        f'{r["Estimativa"]:.4f} '
        f'[{r["IC95_inf"]:.4f}; {r["IC95_sup"]:.4f}]'
    ),
    axis=1
)

tabela_bootstrap = (
    df_final.pivot(
        index=["Modelo", "Grupo"],
        columns="Metrica",
        values="Resultado"
    )
    .reset_index()
)

tabela_bootstrap = tabela_bootstrap[
    ["Modelo", "Grupo", "ScoreSense",
     "LabelDist", "FlipRate", "ScoreDev"]
]

display(tabela_bootstrap)

arquivo_completo = f"BOOTSTRAP_{VERSAO}_COMPLETO.csv"
arquivo_tabela = f"BOOTSTRAP_{VERSAO}_TABELA_ARTIGO.csv"

df_bootstrap.to_csv(
    arquivo_completo,
    index=False,
    encoding="utf-8-sig"
)

tabela_bootstrap.to_csv(
    arquivo_tabela,
    index=False,
    encoding="utf-8-sig"
)

files.download(arquivo_completo)
files.download(arquivo_tabela)

if "df_conferencia" in globals():
    arquivo_validacao = (
        f"BOOTSTRAP_{VERSAO}_VALIDACAO_LABELDIST.csv"
    )

    df_conferencia.to_csv(
        arquivo_validacao,
        index=False,
        encoding="utf-8-sig"
    )

    files.download(arquivo_validacao)

print("Downloads solicitados com sucesso!")

Metrica,Modelo,Grupo,ScoreSense,LabelDist,FlipRate,ScoreDev
0,BERTimbau Toxic,Clinico,0.0427 [0.0091; 0.0756],0.1888 [0.1103; 0.2816],0.0821 [0.0462; 0.1231],0.0478 [0.0266; 0.0550]
1,BERTimbau Toxic,Discurso_Social,0.1812 [0.0063; 0.3495],0.4942 [0.2826; 0.6299],0.3292 [0.1261; 0.5385],0.2004 [0.0729; 0.2279]
2,BERTimbau Toxic,GERAL,0.0577 [-0.0047; 0.1313],0.3478 [0.2266; 0.4675],0.1736 [0.0993; 0.2619],0.1693 [0.0892; 0.2095]
3,BERTimbau Toxic,Neutro,-0.0481 [-0.0934; -0.0027],0.2689 [0.1637; 0.3864],0.1051 [0.0615; 0.1538],0.0704 [0.0344; 0.0816]
4,BERTimbau Toxic,Sem_Deficiencia,0.0847 [-0.0639; 0.3004],0.3986 [0.1404; 0.5960],0.2192 [0.0538; 0.4962],0.1979 [0.0140; 0.2415]
5,BERTweet-PT,Clinico,-0.0653 [-0.1214; -0.0123],0.2946 [0.1772; 0.4207],0.1744 [0.1167; 0.2385],0.0736 [0.0423; 0.0861]
6,BERTweet-PT,Discurso_Social,-0.0551 [-0.1084; -0.0088],0.2442 [0.1429; 0.3575],0.1446 [0.0954; 0.1969],0.0841 [0.0410; 0.0959]
7,BERTweet-PT,GERAL,-0.0007 [-0.0536; 0.0614],0.2883 [0.1892; 0.3977],0.1700 [0.1165; 0.2289],0.1608 [0.0846; 0.2243]
8,BERTweet-PT,Neutro,0.1109 [0.0116; 0.2690],0.3054 [0.1702; 0.4515],0.2026 [0.1013; 0.3462],0.1899 [0.0386; 0.2535]
9,BERTweet-PT,Sem_Deficiencia,-0.0032 [-0.0591; 0.0445],0.3114 [0.1828; 0.4593],0.1462 [0.0962; 0.2038],0.0753 [0.0284; 0.0826]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads solicitados com sucesso!
